# Drive -> SRT (ruso) con `faster-whisper-xxl` (large-v2)

Notebook autocontenido. Levanta videos desde tu Drive (carpeta **Host Videos**), los transcribe a SRT en ruso con **Faster-Whisper-XXL r245.4** (large-v2 + extraccion de voz Kim2) y guarda los `.srt` en **Subs_RU**.

**Antes de correr:** `Runtime -> Change runtime type -> T4 GPU`.

## Dos modos

- **Celda 1 - automatico (cola por Google Sheet).** Las cuentas se reparten el trabajo solas con un Sheet compartido como cola, coherente entre cuentas al instante (a diferencia de Drive). Autoriza los 2 popups al inicio, baja el binario, y se pone a tomar filas. Recomendado para los 500+.
- **Celdas 2 + 3 - manual por rango en bloque** (fallback).

Los dos usan los mismos parametros de Whisper y guardan el SRT con el mismo nombre del video (`pepe.mp4` -> `pepe.srt`).

## Parametros

```
-m large-v2                 -l ru                 --task transcribe
--initial_prompt None       --reprompt False      --condition_on_previous_text False
--hallucination_silence_threshold 4
--compute_type float16      --temperature 0       --beam_size 5
--vad_filter True
--ff_vocal_extract mdx_kim2 --voc_device cuda     --ff_loudnorm
-f srt                      --max_line_width 200  --max_line_count 1   --sentence
```

> Kim2 (separacion de voz) corre en la misma GPU antes de transcribir; baja un modelo la primera vez y sube el tiempo por video ~1.5x-2x.


## 1) Modo automatico - cola por Google Sheet (RECOMENDADO)

Corre **solo esta celda** en cada cuenta. No hay que editar nada entre cuentas.

**Orden de la celda:** autoriza Drive + Sheets (los 2 popups salen **al inicio**, antes de la descarga, asi no esperas) -> baja el binario -> entra al loop.

**El Sheet** (`Subs_RU/Status_Videos`) tiene 5 columnas: `video | status | worker | claim_time | tries`. Cada cuenta:

1. Lee toda la tabla (1 request) y toma la primera fila reclamable cuyo video tenga subido: `TODO`, o `DOING` vencido (>30 min = huerfano), o `FAILED` con menos de 3 intentos.
2. Marca `DOING + worker + claim_time`, espera 3s, relee: si quedo su worker, gano; si no, va a la siguiente.
3. Transcribe, deja el SRT en `Subs_RU`, marca `DONE`. Si ya existia el SRT, marca `DONE` sin reprocesar.
4. **Reintentos:** si falla, suma 1 a `tries` y marca `FAILED`. Otra cuenta lo retoma hasta llegar a 3 intentos; ahi queda `FAILED` definitivo.
5. **Sin trabajo:** pita y espera 60s (util mientras subis tandas). A los **10 chequeos vacios seguidos desconecta el runtime** (`runtime.unassign()`) para no gastar cuota.

> **IMPORTANTE - antes de correr la primera vez:** agrega en el Sheet la columna **`tries`** en **E1** (las celdas E2 en adelante se dejan vacias; el codigo las cuenta como 0). El resto del header ya lo tenes: `video | status | worker | claim_time`.


In [ ]:
import os, time, re, subprocess, shutil, random
from pathlib import Path
from datetime import datetime, timezone

# ===== Autorizaciones PRIMERO: los popups salen al inicio, antes de la descarga =====
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")            # popup 1: Drive
print("Drive montado.")
from google.colab import auth
auth.authenticate_user()                      # popup 2: APIs de Google (Sheets)
get_ipython().system("pip install -q --upgrade gspread")
import gspread
from gspread import Cell
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
SHEET_ID  = "10G9lkcjEY_Ns63zWXDTWIo_xT3th044QSYIdyOAqf0M"
SHEET_TAB = "Hoja 1"
ws = gc.open_by_key(SHEET_ID).worksheet(SHEET_TAB)
print("Sheet conectado.")

# ===== Instalacion del binario (descarga larga; ya autorizaste, podes irte) =====
get_ipython().system("apt-get -qq install -y ffmpeg p7zip-full > /dev/null")
import torch
XXL_URL = "https://github.com/Purfview/whisper-standalone-win/releases/download/Faster-Whisper-XXL/Faster-Whisper-XXL_r245.4_linux.7z"
INSTALL_DIR = Path("/content/whisper")
EXE = INSTALL_DIR / "Faster-Whisper-XXL" / "faster-whisper-xxl"
ARCHIVE = INSTALL_DIR / "fwxxl.7z"
if EXE.exists():
    print(f"Binario ya instalado: {EXE}")
else:
    INSTALL_DIR.mkdir(parents=True, exist_ok=True)
    if not ARCHIVE.exists():
        print("Descargando Faster-Whisper-XXL r245.4 para Linux (~1.54 GB, una sola vez)...")
        rc = subprocess.call(["wget", "-q", "--show-progress", "-O", str(ARCHIVE), XXL_URL])
        assert rc == 0 and ARCHIVE.stat().st_size > 100 * 1024 * 1024, "Descarga incompleta"
    print("Extrayendo (1-2 min)...")
    rc = subprocess.call(["7z", "x", "-y", "-bso0", "-bsp0", f"-o{INSTALL_DIR}", str(ARCHIVE)])
    assert rc == 0 and EXE.exists(), "Extraccion fallo"
    ARCHIVE.unlink(missing_ok=True)
os.chmod(EXE, 0o755)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    print("[WARN] sin GPU: float16 y la extraccion de voz en cuda van a fallar. Activa T4 GPU.")

# ===== Carpetas, listado, comando (compartido) =====
INPUT_DIR  = Path("/content/drive/MyDrive/Host Videos")
OUTPUT_DIR = Path("/content/drive/MyDrive/Subs_RU")
assert INPUT_DIR.exists(), f"No existe la carpeta de entrada: {INPUT_DIR}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_EXTS = {".mp4", ".mkv", ".webm", ".mov", ".avi", ".m4v",
              ".m4a", ".mp3", ".wav", ".ogg", ".opus", ".aac", ".flac"}

def natural_key(p):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", p.name)]

inputs = sorted((p for p in INPUT_DIR.iterdir()
                 if p.is_file() and p.suffix.lower() in VIDEO_EXTS), key=natural_key)
assert inputs, f"No hay archivos de video/audio en {INPUT_DIR}"
total = len(inputs)
already = sum(1 for p in inputs if (OUTPUT_DIR / f"{p.stem}.srt").exists())
print(f"\n{total} archivo(s) en 'Host Videos'  |  {already} con SRT  |  {total-already} pendientes")

TMP_OUT = Path("/content/srt_tmp"); TMP_OUT.mkdir(exist_ok=True)

def build_cmd(vid):
    return [
        str(EXE), str(vid),
        "--model", "large-v2", "--language", "ru", "--task", "transcribe",
        "--initial_prompt", "None", "--reprompt", "False",
        "--condition_on_previous_text", "False",
        "--hallucination_silence_threshold", "4",
        "--compute_type", "float16", "--temperature", "0", "--beam_size", "5",
        "--vad_filter", "True",
        "--ff_vocal_extract", "mdx_kim2", "--voc_device", "cuda", "--ff_loudnorm",
        # "--diarize", "pyannote_v3.1", "--diarize_device", "cuda",
        "--max_line_width", "200", "--max_line_count", "1", "--sentence",
        "--output_dir", str(TMP_OUT), "--output_format", "srt",
    ]

def transcribe_one(vid, label):
    final_srt = OUTPUT_DIR / f"{vid.stem}.srt"
    for f in TMP_OUT.glob(f"{vid.stem}.*"):
        try: f.unlink()
        except OSError: pass
    t0 = time.time()
    result = subprocess.run(build_cmd(vid), capture_output=True, text=True)
    cli_srt = TMP_OUT / f"{vid.stem}.srt"
    if result.returncode == 0 and cli_srt.exists():
        if final_srt.exists():
            cli_srt.unlink(missing_ok=True); return "skipped"
        shutil.move(str(cli_srt), str(final_srt))
        n = sum(1 for L in final_srt.read_text(encoding="utf-8").splitlines() if L.strip().isdigit())
        print(f"   ok {label}: {n} cues, {time.time()-t0:.1f}s -> {final_srt.name}")
        return "done"
    print(f"   x {label}: CLI exit {result.returncode}; sin SRT.")
    err = (result.stderr or "")[-400:]
    if err: print(f"   stderr:\n{err}")
    return "failed"

# ===== Cola coordinada por Google Sheet: beep, auto-apagado y reintentos =====
import numpy as np
from IPython.display import Audio, display

WORKER    = ""      # opcional: nombre de ESTA cuenta. Vacio = autogenerado.
TTL_MIN   = 30      # fila DOING mas vieja que esto = huerfana (reclamable)
VERIFY_S  = 3       # espera entre marcar DOING y releer (confirmar el claim)
IDLE_S    = 60      # espera cuando no hay nada para mi
MAX_IDLE  = 10      # tras esta cantidad de chequeos vacios seguidos, desconecto el runtime
MAX_TRIES = 3       # intentos por video antes de dejarlo FAILED definitivo

if not WORKER:
    import uuid; WORKER = "colab-" + uuid.uuid4().hex[:6]
print(f"Worker: {WORKER}")

COL_STATUS, COL_WORKER, COL_TIME, COL_TRIES = 2, 3, 4, 5   # B, C, D, E

local_stems = {p.stem for p in inputs}

def _now(): return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
def _stem(name):
    name = name.strip()
    for e in VIDEO_EXTS:
        if name.lower().endswith(e): return name[:-len(e)]
    return name
def _orphan(ctime):
    try:
        age = (datetime.now(timezone.utc) -
               datetime.strptime(ctime, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)).total_seconds() / 60
        return age > TTL_MIN
    except Exception:
        return True
def _tries(row): return int(row[4]) if len(row) > 4 and row[4].strip().isdigit() else 0
def beep(freq=880, dur=0.12):
    sr = 8000; t = np.linspace(0, dur, int(sr*dur), endpoint=False)
    display(Audio((0.2*np.sin(2*np.pi*freq*t)).astype(np.float32), rate=sr, autoplay=True))

done = skipped = failed = 0
idle = 0
t_global = time.time()

while True:
    rows = ws.get_all_values()[1:]                 # 1 request: toda la tabla
    target = None
    for i, row in enumerate(rows, start=2):
        video  = row[0] if len(row) > 0 else ""
        status = (row[1] if len(row) > 1 else "").strip().upper()
        ctime  = row[3] if len(row) > 3 else ""
        tries  = _tries(row)
        if not video:                                continue
        if _stem(video) not in local_stems:          continue   # no subido a esta cuenta
        if status == "DONE":                         continue
        if status == "DOING" and not _orphan(ctime): continue   # tomado y vigente
        if status == "FAILED" and tries >= MAX_TRIES: continue   # agoto reintentos
        target = (i, _stem(video), video, tries)
        break

    if target is None:
        pend = 0
        for r in rows:
            if not (r and r[0]): continue
            st = (r[1] if len(r) > 1 else "").strip().upper()
            if st == "DONE": continue
            if st == "FAILED" and _tries(r) >= MAX_TRIES: continue
            pend += 1
        if pend == 0:
            print("\nTodo terminado (DONE, o FAILED tras agotar reintentos). Nada que hacer."); break
        idle += 1
        beep()
        if idle >= MAX_IDLE:
            print(f"\n{MAX_IDLE} chequeos seguidos sin trabajo -> desconecto el runtime para no gastar cuota.")
            from google.colab import runtime; runtime.unassign(); break
        print(f"\nNada para mi ahora ({pend} pendientes, tomadas por otras o no subidas)."
              f" Idle {idle}/{MAX_IDLE}, espero {IDLE_S}s...")
        time.sleep(IDLE_S); continue

    idle = 0
    rownum, stem, video, tries = target

    # Ya hecho de antes (SRT presente): marco DONE y sigo.
    if (OUTPUT_DIR / f"{stem}.srt").exists():
        ws.update_cells([Cell(rownum, COL_STATUS, "DONE")]); skipped += 1; continue

    # Claim: DOING + worker + timestamp.
    ws.update_cells([Cell(rownum, COL_STATUS, "DOING"), Cell(rownum, COL_WORKER, WORKER),
                     Cell(rownum, COL_TIME, _now())])
    time.sleep(VERIFY_S)
    check = ws.row_values(rownum)
    if len(check) < 3 or check[2] != WORKER:        # perdi la carrera
        continue

    print(f"\n[fila {rownum}] (intento {tries+1}/{MAX_TRIES}) Procesando: {video}")
    vid_path = next((p for p in inputs if p.stem == stem), None)
    if vid_path is None:
        ws.update_cells([Cell(rownum, COL_STATUS, "TODO"), Cell(rownum, COL_WORKER, ""),
                         Cell(rownum, COL_TIME, "")]); continue
    try:
        outcome = transcribe_one(vid_path, f"fila {rownum}")
    except Exception as ex:
        print(f"   x excepcion: {ex}"); outcome = "failed"

    if outcome in ("done", "skipped"):
        ws.update_cells([Cell(rownum, COL_STATUS, "DONE"), Cell(rownum, COL_WORKER, WORKER),
                         Cell(rownum, COL_TIME, _now())])
        done += outcome == "done"; skipped += outcome == "skipped"
    else:
        ws.update_cells([Cell(rownum, COL_STATUS, "FAILED"), Cell(rownum, COL_WORKER, WORKER),
                         Cell(rownum, COL_TIME, _now()), Cell(rownum, COL_TRIES, str(tries + 1))])
        failed += 1

print(f"\n=== Fin ({WORKER}) ===")
print(f"  transcritos: {done}  |  saltados: {skipped}  |  fallidos (este run): {failed}")
print(f"  tiempo: {(time.time()-t_global)/60:.1f} min")

sr = 22050; out_audio = np.array([], dtype=np.float32)
for f in [392, 523, 659, 784, 1047]:
    t = np.linspace(0, 0.18, int(sr*0.18), endpoint=False)
    out_audio = np.concatenate([out_audio, (0.3*np.exp(-3*t)*np.sin(2*np.pi*f*t)).astype(np.float32)])
display(Audio(out_audio, rate=sr, autoplay=True))


## 2) Setup (modo manual por rango)

Monta Drive (popup al inicio), baja el binario, lista `Host Videos`, sugiere un reparto y te da un cuadro para el rango (ej. `1-100`). Despues, celda 3.

> Si usas el modo automatico (celda 1), ignora las celdas 2 y 3.


In [ ]:
import os, time, re, subprocess, shutil, random
from pathlib import Path
from datetime import datetime, timezone

# ===== Autorizaciones PRIMERO: los popups salen al inicio, antes de la descarga =====
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")            # popup 1: Drive
print("Drive montado.")

# ===== Instalacion del binario (descarga larga; ya autorizaste, podes irte) =====
get_ipython().system("apt-get -qq install -y ffmpeg p7zip-full > /dev/null")
import torch
XXL_URL = "https://github.com/Purfview/whisper-standalone-win/releases/download/Faster-Whisper-XXL/Faster-Whisper-XXL_r245.4_linux.7z"
INSTALL_DIR = Path("/content/whisper")
EXE = INSTALL_DIR / "Faster-Whisper-XXL" / "faster-whisper-xxl"
ARCHIVE = INSTALL_DIR / "fwxxl.7z"
if EXE.exists():
    print(f"Binario ya instalado: {EXE}")
else:
    INSTALL_DIR.mkdir(parents=True, exist_ok=True)
    if not ARCHIVE.exists():
        print("Descargando Faster-Whisper-XXL r245.4 para Linux (~1.54 GB, una sola vez)...")
        rc = subprocess.call(["wget", "-q", "--show-progress", "-O", str(ARCHIVE), XXL_URL])
        assert rc == 0 and ARCHIVE.stat().st_size > 100 * 1024 * 1024, "Descarga incompleta"
    print("Extrayendo (1-2 min)...")
    rc = subprocess.call(["7z", "x", "-y", "-bso0", "-bsp0", f"-o{INSTALL_DIR}", str(ARCHIVE)])
    assert rc == 0 and EXE.exists(), "Extraccion fallo"
    ARCHIVE.unlink(missing_ok=True)
os.chmod(EXE, 0o755)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    print("[WARN] sin GPU: float16 y la extraccion de voz en cuda van a fallar. Activa T4 GPU.")

# ===== Carpetas, listado, comando (compartido) =====
INPUT_DIR  = Path("/content/drive/MyDrive/Host Videos")
OUTPUT_DIR = Path("/content/drive/MyDrive/Subs_RU")
assert INPUT_DIR.exists(), f"No existe la carpeta de entrada: {INPUT_DIR}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_EXTS = {".mp4", ".mkv", ".webm", ".mov", ".avi", ".m4v",
              ".m4a", ".mp3", ".wav", ".ogg", ".opus", ".aac", ".flac"}

def natural_key(p):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", p.name)]

inputs = sorted((p for p in INPUT_DIR.iterdir()
                 if p.is_file() and p.suffix.lower() in VIDEO_EXTS), key=natural_key)
assert inputs, f"No hay archivos de video/audio en {INPUT_DIR}"
total = len(inputs)
already = sum(1 for p in inputs if (OUTPUT_DIR / f"{p.stem}.srt").exists())
print(f"\n{total} archivo(s) en 'Host Videos'  |  {already} con SRT  |  {total-already} pendientes")

TMP_OUT = Path("/content/srt_tmp"); TMP_OUT.mkdir(exist_ok=True)

def build_cmd(vid):
    return [
        str(EXE), str(vid),
        "--model", "large-v2", "--language", "ru", "--task", "transcribe",
        "--initial_prompt", "None", "--reprompt", "False",
        "--condition_on_previous_text", "False",
        "--hallucination_silence_threshold", "4",
        "--compute_type", "float16", "--temperature", "0", "--beam_size", "5",
        "--vad_filter", "True",
        "--ff_vocal_extract", "mdx_kim2", "--voc_device", "cuda", "--ff_loudnorm",
        # "--diarize", "pyannote_v3.1", "--diarize_device", "cuda",
        "--max_line_width", "200", "--max_line_count", "1", "--sentence",
        "--output_dir", str(TMP_OUT), "--output_format", "srt",
    ]

def transcribe_one(vid, label):
    final_srt = OUTPUT_DIR / f"{vid.stem}.srt"
    for f in TMP_OUT.glob(f"{vid.stem}.*"):
        try: f.unlink()
        except OSError: pass
    t0 = time.time()
    result = subprocess.run(build_cmd(vid), capture_output=True, text=True)
    cli_srt = TMP_OUT / f"{vid.stem}.srt"
    if result.returncode == 0 and cli_srt.exists():
        if final_srt.exists():
            cli_srt.unlink(missing_ok=True); return "skipped"
        shutil.move(str(cli_srt), str(final_srt))
        n = sum(1 for L in final_srt.read_text(encoding="utf-8").splitlines() if L.strip().isdigit())
        print(f"   ok {label}: {n} cues, {time.time()-t0:.1f}s -> {final_srt.name}")
        return "done"
    print(f"   x {label}: CLI exit {result.returncode}; sin SRT.")
    err = (result.stderr or "")[-400:]
    if err: print(f"   stderr:\n{err}")
    return "failed"

# ===== Reparto sugerido + cuadro de rango (modo manual) =====
import ipywidgets as widgets
from IPython.display import display
N_ACCOUNTS = 5
chunk = (total + N_ACCOUNTS - 1) // N_ACCOUNTS
print(f"\nReparto sugerido para {N_ACCOUNTS} cuentas (~{chunk} c/u):")
for k in range(N_ACCOUNTS):
    a = k * chunk + 1; b = min((k + 1) * chunk, total)
    if a > total: break
    print(f"   cuenta {k+1} -> {a}-{b}")
range_box = widgets.Text(value=f"1-{min(100, total)}", placeholder="ej: 1-100 (o 'all')",
                         description="Rango:", layout=widgets.Layout(width="60%"),
                         style={"description_width": "60px"})
print(f"\nElegi el rango (1..{total}) y pasa a la celda 3:")
display(range_box)


## 3) Transcribir un rango (modo manual)

Lee el rango del cuadro de la celda 2 y transcribe solo esos. Salta los que ya tienen SRT.


In [ ]:
import numpy as np
from IPython.display import Audio, display
assert "range_box" in globals(), "Primero corre la celda 2 (setup manual)."
spec = (range_box.value or "").strip().lower()
if spec in ("", "all"):
    start, end = 1, total
else:
    m = re.fullmatch(r"(\d+)\s*-\s*(\d+)", spec) or re.fullmatch(r"(\d+)", spec)
    assert m, f"Rango invalido: {spec!r}. Usa '1-100' o '50' o 'all'."
    start, end = (int(m.group(1)), int(m.group(1))) if m.lastindex == 1 else (int(m.group(1)), int(m.group(2)))
start = max(1, start); end = min(total, end)
assert start <= end, f"Rango vacio tras recortar a 1..{total}: {start}-{end}"
batch = inputs[start-1:end]
print(f"Procesando {len(batch)} archivo(s): #{start} a #{end} de {total}.")
done = skipped = failed = 0; t_global = time.time()
for offset, vid in enumerate(batch):
    i = start + offset
    if (OUTPUT_DIR / f"{vid.stem}.srt").exists():
        print(f"\n[{i}/{end}] SALTADO (ya existe): {vid.stem}.srt"); skipped += 1; continue
    print(f"\n[{i}/{end}] Procesando: {vid.name}")
    try: outcome = transcribe_one(vid, f"{i}/{end}")
    except Exception as ex: print(f"   x excepcion: {ex}"); outcome = "failed"
    done += outcome == "done"; skipped += outcome == "skipped"; failed += outcome == "failed"
print(f"\n=== Rango {start}-{end}: {done} ok, {skipped} saltados, {failed} fallidos, {(time.time()-t_global)/60:.1f} min ===")
sr = 22050; out_audio = np.array([], dtype=np.float32)
for f in [392, 523, 659, 784, 1047]:
    t = np.linspace(0, 0.18, int(sr*0.18), endpoint=False)
    out_audio = np.concatenate([out_audio, (0.3*np.exp(-3*t)*np.sin(2*np.pi*f*t)).astype(np.float32)])
display(Audio(out_audio, rate=sr, autoplay=True))
